# P1 · EEG · 00: Chuẩn bị dữ liệu THINGS-EEG2 trên Colab

Notebook này làm năm việc:
1. Tải dữ liệu THINGS-EEG2 đã tiền xử lý (63 kênh, 250 Hz, bản NICE và ATM dùng) từ Hugging Face.
2. Chuyển từ float64 sang float32 và lưu lên Drive dạng `.npy`, mở được bằng memmap. Code gốc của NICE và ATM đều đổi dữ liệu sang float32 trước khi đưa vào model, nên float32 giữ đúng con số model nhìn thấy.
3. Kiểm tra kích thước, trục thời gian và thứ tự ảnh.
4. Tách validation theo concept từ tập train.
5. Gán 27 category cấp cao cho 200 ảnh test.

**Dung lượng mỗi lần tải** (đọc trước khi chạy):

| File | Dung lượng gốc | Sau khi chuyển float32 |
|---|---|---|
| EEG train, một subject | 8,34 GB | khoảng 4,17 GB |
| EEG test, một subject | 2,02 GB | khoảng 1,01 GB |
| Gói ảnh `images_set.tar.gz` | 0,78 GB | giữ nguyên |
| Đặc trưng CLIP ViT-H-14 của ATM | khoảng 0,07 GB | giữ nguyên |

Mặc định notebook chỉ xử lý **sub-01**. Khi đã kiểm tra xong, chị đổi danh sách `SUBJECTS` để chạy các subject còn lại.

**Drive:** 10 subject ở float32 chiếm khoảng 52 GB, cần gói Google One 100 GB.

**Runtime:** bước chuyển đổi cần khoảng 9–10 GB RAM, không cần GPU. Nếu runtime thường bị tràn RAM, chọn High-RAM.

## 1. Nối Google Drive và chọn thư mục

Mọi dữ liệu và kết quả lâu dài nằm trên Drive. Ổ `/content` của Colab bị xoá khi phiên kết thúc, nên chỉ dùng làm chỗ tạm.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/BrainDecoding')   # đổi nếu chị muốn để chỗ khác
DATA = DRIVE / 'data'
EEG_DIR = DATA / 'things_eeg2' / 'eeg_63ch_250hz_f32'  # float32: đúng kiểu mà code NICE và ATM đưa vào model
META_DIR = DATA / 'things_meta'
RESULTS = DRIVE / 'results'
LOGS = DRIVE / 'logs'
for p in [EEG_DIR, META_DIR, RESULTS, LOGS, DATA / 'things_eeg2' / 'archives']:
    p.mkdir(parents=True, exist_ok=True)
SCRATCH = Path('/content/scratch'); SCRATCH.mkdir(exist_ok=True)
print('Drive:', DRIVE)

## 2. Lấy code

Code lấy từ repo private `ThanhKieuVT/brain-decoding`. Lần đầu notebook clone repo; các lần sau tự `git pull` để lấy bản mới nhất.

Cần một token chỉ đọc: trên GitHub vào Settings, Developer settings, Fine-grained tokens, tạo token chỉ cho repo `brain-decoding` với quyền Contents: Read-only. Trên Colab, mở mục Secrets (biểu tượng chìa khoá bên trái), thêm secret tên `GH_TOKEN` và bật quyền cho notebook.

Nếu không dùng được GitHub, đổi `GET_CODE = 'zip'` và tải file zip tạo bằng `bash scripts/pack_for_colab.sh` lên `MyDrive/BrainDecoding/code/`.

In [ ]:
import os, subprocess, zipfile
REPO = Path('/content/brain_decoding')
GET_CODE = 'git'   # 'git' hoặc 'zip'
GIT_URL = 'github.com/ThanhKieuVT/brain-decoding.git'

if GET_CODE == 'git':
    from google.colab import userdata
    try:
        token = userdata.get('GH_TOKEN')
    except Exception as e:
        raise RuntimeError('Chưa có secret GH_TOKEN trong Colab, hoặc chưa bật quyền cho notebook') from e
    url = f'https://{token}@{GIT_URL}'
    if (REPO / '.git').exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '-q', url, 'main'], check=True)
        print('Đã cập nhật code mới nhất')
    else:
        subprocess.run(['git', 'clone', '-q', url, str(REPO)], check=True)
        print('Đã clone code')
elif REPO.exists():
    print('Code đã có ở', REPO)
else:
    zips = sorted((DRIVE / 'code').glob('brain_decoding_*.zip'))
    assert zips, 'Chưa có file zip trong MyDrive/BrainDecoding/code/'
    zipfile.ZipFile(zips[-1]).extractall(REPO)
    print('Đã giải nén', zips[-1].name)
os.chdir(REPO)
import sys; sys.path.insert(0, str(REPO))

In [ ]:
# Cài các gói còn thiếu. Không cài lại torch: dùng bản CUDA sẵn có của Colab.
!pip install -q -r requirements-colab.txt
!python -m pytest -q 2>&1 | tail -2

In [ ]:
# Ghi cấu hình máy và phiên bản gói lên Drive để tái lập được kết quả
import datetime, json, torch, platform, psutil
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
info = {
    'time': stamp, 'python': platform.python_version(), 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'gpu': gpu,
    'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if gpu != 'none' else 0,
    'ram_gb': round(psutil.virtual_memory().total / 1e9, 1),
    'disk_free_gb_content': round(psutil.disk_usage('/content').free / 1e9, 1),
    'commit': subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'zip',
}
print(info)
(LOGS / f'colab_env_{stamp}.json').write_text(json.dumps(info, indent=1))
!pip freeze > {LOGS}/pip_freeze_{stamp}.txt

## 3. Tải dữ liệu nhỏ: ảnh, đặc trưng CLIP của ATM, metadata THINGS

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, urllib.request
HF_REPO = 'LidongYang/EEG_Image_decode'
ARCH = DATA / 'things_eeg2' / 'archives'

def hf_get(filename, dest_dir):
    dest = Path(dest_dir) / Path(filename).name
    if dest.exists():
        print('Đã có:', dest.name); return dest
    p = hf_hub_download(HF_REPO, filename, repo_type='dataset', local_dir=SCRATCH / 'hf')
    shutil.copy(p, dest); print('Đã tải:', dest.name, round(dest.stat().st_size / 1e9, 2), 'GB')
    return dest

img_tar = hf_get('images_set.tar.gz', ARCH)
hf_get('ViT-H-14_features_train.pt', ARCH)
hf_get('ViT-H-14_features_test.pt', ARCH)

# Metadata THINGS (OSF dự án jum2f): 27 category và danh sách concept
for name, url in [('category27_manual.tsv', 'https://osf.io/download/5mvxh/'),
                  ('category27_top-down.tsv', 'https://osf.io/download/fpq9v/'),
                  ('category27_bottom-up.tsv', 'https://osf.io/download/pc98z/'),
                  ('concepts-metadata_things.tsv', 'https://osf.io/download/um6a9/')]:
    dest = META_DIR / name
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    print(name, dest.stat().st_size, 'bytes')

In [ ]:
# Giải nén ảnh vào ổ tạm (giữ file tar trên Drive; Drive chậm với hàng chục nghìn file nhỏ)
import tarfile
IMG_ROOT = SCRATCH / 'images'
if not IMG_ROOT.exists():
    with tarfile.open(img_tar) as t:
        t.extractall(IMG_ROOT)
from brain.data.things_eeg2 import find_condition_dirs, list_image_conditions
dirs = find_condition_dirs(IMG_ROOT)
TRAIN_IMG, TEST_IMG = dirs[1654], dirs[200]
train_concepts = list_image_conditions(TRAIN_IMG)
test_concepts = list_image_conditions(TEST_IMG)
print(TRAIN_IMG, len(train_concepts)); print(TEST_IMG, len(test_concepts))
print('Ảnh train:', sum(len(list((TRAIN_IMG / c).iterdir())) for c in train_concepts))

## 4. Tải và chuyển đổi EEG theo từng subject

Mỗi subject: tải hai file gốc vào ổ tạm, chuyển sang float16 ghi lên Drive, rồi xoá file gốc. Bảng `profile.csv` ghi thời gian và RAM đỉnh của bước này.

In [ ]:
from brain.data.things_eeg2 import convert_subject
from bench.profile import Profiler
SUBJECTS = ['sub-01']          # thêm 'sub-02', ... sau khi sub-01 ổn
DTYPE = 'float32'   # 'float16' chỉ dùng khi thiếu chỗ; khi đó không còn khớp số với bài gốc
PROFILE_CSV = RESULTS / 'pilot' / 'profile.csv'

for sub in SUBJECTS:
    out = EEG_DIR / sub
    if (out / 'training_eeg.npy').exists() and (out / 'test_meta.json').exists():
        print(sub, 'đã chuyển đổi, bỏ qua'); continue
    raw = SCRATCH / 'hf' / 'Preprocessed_data_250Hz' / sub
    for f in ['preprocessed_eeg_test.npy', 'preprocessed_eeg_training.npy']:
        hf_hub_download(HF_REPO, f'Preprocessed_data_250Hz/{sub}/{f}', repo_type='dataset',
                        local_dir=SCRATCH / 'hf')
    with Profiler('convert_eeg', n_items=16540 + 200, modality='eeg', subject=sub, stage='convert',
                  data_dir=out, notes=DTYPE, csv_path=PROFILE_CSV):
        report = convert_subject(raw, out, dtype=DTYPE)
    print(sub, json.dumps(report, indent=1))
    shutil.rmtree(raw)            # xoá bản float64 để giải phóng ổ tạm
    shutil.rmtree(SCRATCH / 'hf' / '.cache', ignore_errors=True)

In [ ]:
# Kiểm tra: kích thước, trục thời gian, kênh
from brain.data.things_eeg2 import open_split
for sub in SUBJECTS:
    tr, te = open_split(EEG_DIR / sub, 'training'), open_split(EEG_DIR / sub, 'test')
    print(sub, 'train', tr.data.shape, 'test', te.data.shape, tr.data.dtype)
    print('  times: %.3f đến %.3f s, %d mẫu; %d kênh' % (tr.times[0], tr.times[-1], len(tr.times), len(tr.ch_names)))
    assert tr.data.shape[0] == len(train_concepts) * 10 and te.data.shape[0] == len(test_concepts)
    meta = json.loads((EEG_DIR / sub / 'training_meta.json').read_text())
    print('  sai số đổi kiểu lớn nhất: %.2e (giá trị lớn nhất %.2f)' % (
        meta['max_abs_conversion_error_first_rows'], meta['max_abs_value_first_rows']))

### Kiểm tra giao thức của NICE và ATM trên dữ liệu thật

ATM cắt mảng thời gian thành `times[50:]` rồi dùng làm mặt nạ cho trục thời gian của dữ liệu. Ô dưới cho biết phép cắt này có hợp lệ trên file thật không. Nếu báo lỗi, chị gửi em đoạn in ra, em sẽ đọc lại code ATM trước khi tái hiện.

In [ ]:
from brain.protocols import NICE, ATM, prepare_eeg
sub = SUBJECTS[0]
tr = open_split(EEG_DIR / sub, 'training')
print('len(times) =', len(tr.times), '| số mẫu thời gian trong dữ liệu =', tr.data.shape[-1])
print('times đầu/cuối:', tr.times[:3], tr.times[-3:])
x, idx = prepare_eeg(tr.data[:20], tr.times, NICE, 'train')
print('NICE train (20 ảnh đầu):', x.shape)
try:
    x, idx = prepare_eeg(tr.data[:20], tr.times, ATM, 'train')
    print('ATM train (20 ảnh đầu):', x.shape, '-> phép cắt thời gian của ATM hợp lệ')
except ValueError as e:
    print('ATM:', e)

## 5. Tách validation theo concept

74 concept × 10 ảnh = 740 ảnh validation, cùng cỡ với NICE nhưng tách theo concept để giống bài toán zero-shot. File được lưu một lần và dùng lại cho mọi decoder, mọi seed.

In [ ]:
from brain.data.splits import concept_val_split, ConceptSplit
SPLIT_FILE = DATA / 'things_eeg2' / 'val_split_seed0.json'
if SPLIT_FILE.exists():
    split = ConceptSplit.load(SPLIT_FILE)
else:
    split = concept_val_split(n_concepts=len(train_concepts), n_val=74, seed=0)
    split.save(SPLIT_FILE)
print('val concept:', len(split.val_concepts), '| train concept:', len(split.train_concepts))
print('ví dụ concept validation:', [train_concepts[i] for i in split.val_concepts[:5]])

## 6. Gán 27 category cấp cao cho 200 ảnh test

Trong THINGS, một concept có thể thuộc 0, 1 hoặc nhiều category. Mặc định, concept thuộc nhiều category bị loại khỏi shuf-w. Cách xử lý này là một lựa chọn thiết kế: chị xem bảng đếm bên dưới rồi chốt.

In [ ]:
import pandas as pd
from brain.data.categories import load_category27, assign_categories
from brain.data.things_eeg2 import concept_id_from_folder
test_ids = [concept_id_from_folder(c) for c in test_concepts]
rows = []
for variant in ['manual', 'top-down', 'bottom-up']:
    names, mapping = load_category27(META_DIR / f'category27_{variant}.tsv', META_DIR / 'concepts-metadata_things.tsv')
    for policy in ['exclude', 'first']:
        labels, summ = assign_categories(test_ids, names, mapping, multi=policy)
        rows.append({'variant': variant, 'multi': policy, 'no_category': summ['no_category'],
                     'multi_category': summ['multi_category'], 'usable_for_shuf_w': summ['usable_for_shuf_w'],
                     'n_categories_with_2plus': sum(v >= 2 for v in summ['per_category'].values())})
cmp = pd.DataFrame(rows); display(cmp)

# Bảng chi tiết cho lựa chọn mặc định
names, mapping = load_category27(META_DIR / 'category27_manual.tsv', META_DIR / 'concepts-metadata_things.tsv')
labels, summ = assign_categories(test_ids, names, mapping, multi='exclude')
print(json.dumps(summ['per_category'], indent=1, ensure_ascii=False))
pd.DataFrame({'image_index': range(len(test_ids)), 'folder': test_concepts, 'concept': test_ids,
              'categories': ['|'.join(mapping[c]) for c in test_ids], 'label_exclude_multi': labels}
            ).to_csv(RESULTS / 'test_categories_manual.csv', index=False)
cmp.to_csv(RESULTS / 'test_category_policy_counts.csv', index=False)
print('Đã lưu', RESULTS / 'test_categories_manual.csv')

## Xong

Trên Drive giờ có:
- `data/things_eeg2/eeg_63ch_250hz_f32/sub-XX/`: EEG float32 và file meta.
- `data/things_eeg2/archives/`: gói ảnh và đặc trưng CLIP của ATM.
- `data/things_eeg2/val_split_seed0.json`: danh sách concept validation.
- `results/test_categories_manual.csv` và `results/test_category_policy_counts.csv`: category của ảnh test.
- `results/pilot/profile.csv`: thời gian và RAM của bước chuyển đổi.

Chị gửi lại cho em hai file CSV về category và dòng trong `profile.csv` để em chốt bước tiếp theo.